# Module 04.06 — Dashboards (`dashboard`)

> Part of the **Elasticsearch Snapshot Training** course.
> Run `00_setup.ipynb` first to start the Docker stack and load sample data.


In [ ]:
import sys, json, time
sys.path.insert(0, '..')
from helpers import *

es = get_client()

---
## 4.6 Dashboards (`dashboard`)

A dashboard is a **container of panels**, not a standalone visualization. Each panel on a
dashboard is a reference to another saved object — a Lens chart, a legacy visualization,
a Map, a saved search, or a Markdown widget — plus layout metadata (position, size) stored
in `panelsJSON`. This makes the dashboard object itself relatively lightweight; it delegates
all the rendering logic to the objects it references.

The `references` array is the key to understanding dashboard restore. Every panel that
points to an external saved object appears as a reference entry. When Kibana restores a
feature state snapshot, it must restore the referenced objects *before* the dashboard,
otherwise the panels will render as "object not found" errors. The feature-state mechanism
handles this ordering automatically.

A common pitfall in partial restores: if you restore only the dashboard object (e.g. by
importing it via the Saved Objects UI without its dependencies), you end up with a
dashboard that renders correctly for objects still present in Kibana but shows errors for
any panel whose source was deleted or never created. Always use the full feature-state
restore — or the "include related objects" option in the Saved Objects import UI — to
keep the graph intact.

Controls (time slider, option lists, range sliders) added to a dashboard are stored
inline in the dashboard's own JSON since Kibana 8.3, rather than as separate saved
objects, which simplifies the reference graph.

### Create in Kibana UI
1. Go to **Dashboards → Create dashboard**
2. Click **Add from library** and add the eCommerce visualizations
3. Arrange panels and save as `Training — eCommerce Overview`

In [ ]:
heading('4.6 Dashboard — inspect saved objects')

dashboards = find_saved_objects('dashboard')
console.print(f'  Found {len(dashboards)} dashboard(s)')
if dashboards:
    d = dashboards[0]
    attrs = d['attributes']
    console.print(f"  Title: {attrs.get('title')}")

    try:
        panels = json.loads(attrs.get('panelsJSON', '[]'))
        console.print(f'  Panels: {len(panels)}')
        for p in panels[:3]:
            console.print(f"    panel {p.get('panelIndex')}: type={p.get('type')}  embeddableConfig={list(p.get('embeddableConfig', {}).keys())}")
    except Exception:
        pass

    console.print(f'  References: {len(d["references"])} linked objects')
    for ref in d['references'][:5]:
        console.print(f"    {ref['type']} : {ref['id']} (alias={ref['name']})")

    info('Key fields:')
    info('  panelsJSON      — layout + embeddable config for each panel')
    info('  optionsJSON     — use margins, sync colors, sync cursor, etc.')
    info('  kibanaSavedObjectMeta.searchSourceJSON — dashboard-level filter')
    info('references        — the critical link graph: panels → vizzes → data views')

In [ ]:
# Visualise the reference graph
heading('Dashboard reference graph')

if dashboards:
    d = dashboards[0]
    from rich.tree import Tree
    tree = Tree(f"[bold cyan]dashboard[/bold cyan] → {d['attributes']['title']} ({d['id'][:8]}...)")
    for ref in d['references']:
        tree.add(f"[yellow]{ref['type']}[/yellow] → {ref['id'][:8]}...  ({ref['name']})")
    console.print(tree)

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
# Open the dashboard list, then click any dashboard to inspect its panels:
kibana_link('/app/dashboards', 'Dashboards — open the dashboard list')
# Or jump directly to the first sample dashboard:
if dashboards:
    d = dashboards[0]
    kibana_link(f'/app/dashboards#/view/{d["id"]}',
                f'Open dashboard: {d["attributes"]["title"]}')

In [ ]:
heading('4.6 Dashboard — create via API with real panels')

# Collect training saved objects to use as panels.
# Run 04_04_lens_visualizations and 04_02_saved_searches first so these exist.
lens_obj = next((o for o in find_saved_objects('lens') if o['attributes'].get('title') == 'Training — Revenue by Day'), None)
search_obj = next((o for o in find_saved_objects('search') if "Men's Clothing" in o['attributes'].get('title', '')), None)

if not lens_obj and not search_obj:
    raise RuntimeError('No training objects found. Run 04_04_lens_visualizations.ipynb and 04_02_saved_searches.ipynb first.')

# Build panels and references from whatever training objects exist
panels, refs = [], []
x = 0
if lens_obj:
    panels.append({'version': '9.3.0', 'type': 'lens', 'panelIndex': 'panel-0',
        'gridData': {'x': 0, 'y': 0, 'w': 36, 'h': 15, 'i': 'panel-0'},
        'embeddableConfig': {'enhancements': {}}, 'panelRefName': 'panel-0'})
    refs.append({'name': 'panel-0', 'type': 'lens', 'id': lens_obj['id']})
    x = 36
if search_obj:
    panels.append({'version': '9.3.0', 'type': 'search', 'panelIndex': 'panel-1',
        'gridData': {'x': x, 'y': 0, 'w': 48 - x, 'h': 15, 'i': 'panel-1'},
        'embeddableConfig': {'enhancements': {}}, 'panelRefName': 'panel-1'})
    refs.append({'name': 'panel-1', 'type': 'search', 'id': search_obj['id']})

# Idempotent: reuse if already exists
existing_dash = next((o for o in find_saved_objects('dashboard') if o['attributes'].get('title') == 'Training — Summary Dashboard'), None)
if existing_dash:
    dash_id = existing_dash['id']
    info(f'Dashboard already exists: id={dash_id} — updating panels')
    import requests as req
    req.put(f'{KIBANA_HOST}/api/saved_objects/dashboard/{dash_id}',
        auth=('elastic', KIBANA_PASSWORD),
        headers={'kbn-xsrf': 'true', 'Content-Type': 'application/json'},
        json={'attributes': {
            'title': 'Training — Summary Dashboard',
            'description': 'Training dashboard for snapshot restore exercise',
            'panelsJSON': json.dumps(panels),
            'optionsJSON': json.dumps({'useMargins': True, 'syncColors': False, 'hidePanelTitles': False}),
            'timeRestore': False,
            'kibanaSavedObjectMeta': {'searchSourceJSON': json.dumps({'query': {'query': '', 'language': 'kuery'}, 'filter': []})},
        }, 'references': refs},
    ).raise_for_status()
else:
    dash_resp = kibana_post('/api/saved_objects/dashboard', {
        'attributes': {
            'title': 'Training — Summary Dashboard',
            'description': 'Training dashboard for snapshot restore exercise',
            'panelsJSON': json.dumps(panels),
            'optionsJSON': json.dumps({'useMargins': True, 'syncColors': False, 'hidePanelTitles': False}),
            'timeRestore': False,
            'kibanaSavedObjectMeta': {'searchSourceJSON': json.dumps({'query': {'query': '', 'language': 'kuery'}, 'filter': []})},
        },
        'references': refs,
    })
    dash_id = dash_resp['id']
    success(f'Dashboard created: id={dash_id}')

obj = kibana_get(f'/api/saved_objects/dashboard/{dash_id}')
pp(obj, 'dashboard saved object')
info('Key fields:')
info('  panelsJSON  — panel layout grid + panelRefName for each panel')
info('  references  — resolves panelRefName → actual saved object (type + id)')
info('  Notice: the dashboard itself is lightweight; all data lives in referenced objects')
kibana_link(f'/app/dashboards#/view/{dash_id}', 'Open Training — Summary Dashboard')

In [ ]:
# Ensure dashboard exists before snapshotting (re-runnable)
if not any(o['id'] == dash_id for o in find_saved_objects('dashboard')):
    warn('Dashboard missing — recreating')
    dash_resp = kibana_post('/api/saved_objects/dashboard', {
        'attributes': {
            'title': 'Training — Summary Dashboard', 'description': 'Training dashboard',
            'panelsJSON': '[]',
            'optionsJSON': json.dumps({'useMargins': True, 'syncColors': False, 'hidePanelTitles': False}),
            'timeRestore': False,
            'kibanaSavedObjectMeta': {'searchSourceJSON': json.dumps({'query': {'query': '', 'language': 'kuery'}, 'filter': []})},
        },
        'references': [],
    })
    dash_id = dash_resp['id']

snap_delete_restore_cycle('snap-dash-test', 'Dashboard')

kibana_delete(f'/api/saved_objects/dashboard/{dash_id}')
warn(f'Accidentally deleted dashboard {dash_id}')
assert not any(o['id'] == dash_id for o in find_saved_objects('dashboard')), 'Should be gone'

restore_kibana_state(es, SNAP_REPO, 'snap-dash-test')
time.sleep(3)

restored = find_saved_objects('dashboard')
assert any(o['id'] == dash_id for o in restored), 'Dashboard should be restored'
success(f'Dashboard {dash_id} successfully restored!')
kibana_link(f'/app/dashboards#/view/{dash_id}', 'Verify restored dashboard')